In [1]:
# prompt: mount colab

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
!pip install -U transformers datasets peft trl accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.3/365.3 kB 12.4 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.7.0
    Uninstalling accelerate-1.7.0:
      Successfully uninstalled accelerate-1.7.0


In [2]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

XLSX_FILE = "/content/drive/My Drive/Inclusive Language Guidleine/inclusive_data.xlsx"
TEXT_COLS = ("non_inclusive", "inclusive")
USE_8BIT = True  # Use 4bit if needed for larger models
BATCH_SIZE = 4  # Reduce if OOM

def load_and_split_data(filepath):
    df = pd.read_excel(filepath)
    dataset = Dataset.from_pandas(df[list(TEXT_COLS)])
    dataset = dataset.map(lambda x: {
        "prompt": f"Rewrite this to be inclusive:\n{x[TEXT_COLS[0]]}",
        "completion": x[TEXT_COLS[1]]
    })
    return dataset.train_test_split(test_size=0.1)

def tokenize(example, tokenizer):
    full_input = f"{example['prompt']}\nInclusive Version: {example['completion']}"
    return tokenizer(full_input, truncation=True, padding="max_length", max_length=512)


In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model

XLSX_FILE = "/content/drive/My Drive/Inclusive Language Guidleine/inclusive_data.xlsx"
TEXT_COLS = ("non_inclusive", "inclusive")
BATCH_SIZE = 1  # CPU memory is limited; keep this very low

def load_and_split_data(filepath):
    df = pd.read_excel(filepath)
    dataset = Dataset.from_pandas(df[list(TEXT_COLS)])
    dataset = dataset.map(lambda x: {
        "prompt": f"Rewrite this to be inclusive:\n{x[TEXT_COLS[0]]}",
        "completion": x[TEXT_COLS[1]]
    })
    return dataset.train_test_split(test_size=0.1)

def tokenize(example, tokenizer):
    full_input = f"{example['prompt']}\nInclusive Version: {example['completion']}"
    return tokenizer(full_input, truncation=True, padding="max_length", max_length=512)

MODEL_NAME = "Qwen/Qwen1.5-1.8B"
OUTPUT_DIR = "./qwen1.5_inclusive_cpu"

def train_qwen_cpu():
    dataset = load_and_split_data(XLSX_FILE)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    dataset = dataset.map(lambda x: tokenize(x, tokenizer), batched=False)

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype="float32",  # Full precision
        trust_remote_code=True
    )

    # Optional: Comment LoRA if you want to run full model without PEFT
    lora_config = LoraConfig(
        r=8, lora_alpha=16, lora_dropout=0.05, target_modules=["q_proj", "v_proj"]
    )
    model = get_peft_model(model, lora_config)

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=1,  # Keep small for testing on CPU
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        logging_dir="./logs_qwen_cpu",
        logging_steps=1,
        no_cuda=True,  # Run on CPU
        save_total_limit=1
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["test"],
        tokenizer=tokenizer,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
    )

    trainer.train()
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)

train_qwen_cpu()


Map:   0%|          | 0/3920 [00:00<?, ? examples/s]

Map:   0%|          | 0/3528 [00:00<?, ? examples/s]

Map:   0%|          | 0/392 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1577: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(
/tmp/ipython-input-1-3643588016.py:61: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModel`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter: